# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
from IPython.display import display, Markdown

display(Markdown("""
## Method Choice

### Selected method: Random Forest

I will use a **Random Forest classifier** as the first capstone model.

### Why it fits this lane

The lane is **content refresh prioritization**, where multiple search and
engagement signals can interact.

Random Forest is a suitable baseline because:

- It can model non-linear relationships between features.
- It can combine multiple signals without requiring a simple linear relationship.
- It works with mixed patterns in tabular data.
- It provides feature importance that can help explain which signals the model
  relies on.

The model will be treated as **decision-support**, not as an automatic
content-refresh decision maker.

### What I will compare

The Random Forest model will be evaluated against the **Week-4 rule-based
baseline** using the same data, split, and evaluation metric.
"""))


## Method Choice

### Selected method: Random Forest

I will use a **Random Forest classifier** as the first capstone model.

### Why it fits this lane

The lane is **content refresh prioritization**, where multiple search and
engagement signals can interact.

Random Forest is a suitable baseline because:

- It can model non-linear relationships between features.
- It can combine multiple signals without requiring a simple linear relationship.
- It works with mixed patterns in tabular data.
- It provides feature importance that can help explain which signals the model
  relies on.

The model will be treated as **decision-support**, not as an automatic
content-refresh decision maker.

### What I will compare

The Random Forest model will be evaluated against the **Week-4 rule-based
baseline** using the same data, split, and evaluation metric.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
from IPython.display import display, Markdown

display(Markdown("""
## Split Design

### Time-aware split

I will use a **time-aware train/test split**.

The model will be trained on earlier observations and evaluated on later
observations.

### Why this is honest

The actual decision is about prioritizing content using information that would
have been available at the time of the decision.

A random split could allow observations from the same time period to appear in
both training and testing data, making the evaluation less representative of
how the model would perform on future content-performance observations.

A time-aware split better reflects the real workflow:

**past data → train model → later data → evaluate model**

The same split will be used when comparing the model with the Week-4 baseline.
"""))


## Split Design

### Time-aware split

I will use a **time-aware train/test split**.

The model will be trained on earlier observations and evaluated on later
observations.

### Why this is honest

The actual decision is about prioritizing content using information that would
have been available at the time of the decision.

A random split could allow observations from the same time period to appear in
both training and testing data, making the evaluation less representative of
how the model would perform on future content-performance observations.

A time-aware split better reflects the real workflow:

**past data → train model → later data → evaluate model**

The same split will be used when comparing the model with the Week-4 baseline.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
import os
import duckdb
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from IPython.display import display, Markdown

# Download the same sample dataset used by the Week-4 baseline
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

con = duckdb.connect()

# Load the same signals used by the baseline
df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-06-01' AND DATE '2026-06-30'
""").df()

# Remove duplicate observations at the content/day/client grain
df = df.drop_duplicates(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).copy()

# Create the same baseline score from ML-07
df["baseline_score"] = (
    df["gsc_impressions"].fillna(0) * 0.40 +
    df["gsc_clicks"].fillna(0) * 0.20 +
    df["ga4_sessions"].fillna(0) * 0.20 +
    df["ga4_engaged_sessions"].fillna(0) * 0.10 +
    df["gsc_avg_position"].fillna(999).apply(
        lambda x: 10 if 1 <= x <= 10 else (5 if 11 <= x <= 20 else 0)
    )
)

# Features
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = df[feature_cols].copy()
X["gsc_avg_position"] = X["gsc_avg_position"].fillna(
    X["gsc_avg_position"].median()
)
X = X.fillna(0)

# Use the baseline score as the measured ranking signal for this comparison.
# This is not a future label.
y = df["baseline_score"]

# Time-aware split: earlier observations for training, later observations for testing
df["_date"] = pd.to_datetime(df["report_date"])

split_date = df["_date"].sort_values().iloc[int(len(df) * 0.80)]

train_mask = df["_date"] < split_date
test_mask = df["_date"] >= split_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

model.fit(X_train, y_train)

model_predictions = model.predict(X_test)

# Compare predictions with the baseline score
mae = mean_absolute_error(y_test, model_predictions)

comparison = pd.DataFrame({
    "Metric": [
        "Test rows",
        "Baseline mean score",
        "Model mean prediction",
        "Model MAE"
    ],
    "Value": [
        len(X_test),
        round(y_test.mean(), 2),
        round(model_predictions.mean(), 2),
        round(mae, 2)
    ]
})

display(Markdown("## Model vs Baseline"))
display(comparison)

# Feature importance
importance = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(Markdown("### Feature Importance"))
display(importance)

## Model vs Baseline

,Metric,Value
0,Test rows,902365.00
1,Baseline mean score,26.81
2,Model mean prediction,26.70
3,Model MAE,0.34


### Feature Importance

,Feature,Importance
0,gsc_impressions,0.988121
3,ga4_sessions,0.007181
2,gsc_avg_position,0.002159
1,gsc_clicks,0.001926
4,ga4_engaged_sessions,0.000614


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
from IPython.display import display, Markdown

# Build an error-analysis table
error_analysis = df.loc[test_mask, [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "baseline_score"
]].copy()

error_analysis["model_prediction"] = model_predictions
error_analysis["absolute_error"] = (
    error_analysis["baseline_score"]
    - error_analysis["model_prediction"]
).abs()

# Largest disagreements between model and baseline
largest_errors = (
    error_analysis
    .sort_values("absolute_error", ascending=False)
    .head(10)
)

display(Markdown("""
## Errors and Interpretation

The model has a low overall MAE of 0.34, meaning its predictions are generally
close to the baseline score.

However, the feature-importance results show that **gsc_impressions accounts
for approximately 98.8% of model importance**. This means the model relies
heavily on search volume and adds relatively little independent signal from the
other features.

The largest disagreements are reviewed below. These cases are useful for
checking whether the model behaves sensibly or whether it is simply reproducing
the volume-driven baseline.

A weakness is that this experiment compares the model with the existing
baseline score rather than a verified future outcome. Therefore, a low MAE
should be interpreted as agreement with the baseline, **not proof that the
model improves future content-refresh decisions**.
"""))

display(largest_errors)


## Errors and Interpretation

The model has a low overall MAE of 0.34, meaning its predictions are generally
close to the baseline score.

However, the feature-importance results show that **gsc_impressions accounts
for approximately 98.8% of model importance**. This means the model relies
heavily on search volume and adds relatively little independent signal from the
other features.

The largest disagreements are reviewed below. These cases are useful for
checking whether the model behaves sensibly or whether it is simply reproducing
the volume-driven baseline.

A weakness is that this experiment compares the model with the existing
baseline score rather than a verified future outcome. Therefore, a low MAE
should be interpreted as agreement with the baseline, **not proof that the
model improves future content-refresh decisions**.


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,baseline_score,model_prediction,absolute_error
3699042,2026-06-29,client_e547b89c05043229,content_eadb33b5df496f4a,49373,173,2.627995,96,4,19813.4,13994.795455,5818.604545
3651977,2026-06-30,client_e547b89c05043229,content_eadb33b5df496f4a,48990,189,2.590467,99,3,19663.9,13933.696455,5730.203545
3550635,2026-06-26,client_e547b89c05043229,content_545bb6cc7081ded3,48953,114,2.586277,52,3,19624.7,14114.229000,5510.471
3877299,2026-06-30,client_06d356715a8ff3b6,content_f88878f155e4838d,45890,329,5.828089,413,0,18514.4,14419.225455,4095.174545
3785689,2026-06-29,client_06d356715a8ff3b6,content_f88878f155e4838d,35560,313,5.676069,377,0,14372.0,10527.868621,3844.131379
3609440,2026-06-25,client_e547b89c05043229,content_545bb6cc7081ded3,42474,107,2.605735,57,4,17032.8,14111.352000,2921.448
3313422,2026-06-27,client_e547b89c05043229,content_545bb6cc7081ded3,41051,129,2.605150,60,1,16468.3,14107.617000,2360.683
3017750,2026-06-24,client_e547b89c05043229,content_0ec99ef7d7e11565,30645,50,5.019710,43,0,12286.6,10266.373667,2020.226333
3278437,2026-06-28,client_e547b89c05043229,content_eadb33b5df496f4a,28952,134,2.513816,88,4,11635.6,10128.264333,1507.335667
3154685,2026-06-28,client_06d356715a8ff3b6,content_f88878f155e4838d,28945,182,5.951736,228,0,11670.0,10520.760955,1149.239045


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [5]:
from IPython.display import display, Markdown

checks = {
    "No future-window features used": True,
    "No label-derived target used": True,
    "Time-aware split used": True,
    "Same baseline signals used for comparison": True,
    "Model evaluated on held-out data": True,
    "Feature importance inspected": True,
    "Largest errors reviewed": True
}

check_df = pd.DataFrame(
    list(checks.items()),
    columns=["Self-check", "PASS"]
)

display(Markdown("## Self-check"))
display(check_df)

assert all(checks.values())

display(Markdown("""
### Conclusion

The Random Forest successfully reproduces the Week-4 baseline scoring pattern
with a low MAE of 0.34.

However, the model is overwhelmingly driven by `gsc_impressions`, and the
experiment does not establish that the model improves future refresh decisions.

The next improvement should therefore focus on obtaining a legitimate
future-window outcome/label and testing whether the model adds value beyond
the hand-written baseline.
"""))

## Self-check

,Self-check,PASS
0,No future-window features used,True
1,No label-derived target used,True
2,Time-aware split used,True
3,Same baseline signals used for comparison,True
4,Model evaluated on held-out data,True
5,Feature importance inspected,True
6,Largest errors reviewed,True



### Conclusion

The Random Forest successfully reproduces the Week-4 baseline scoring pattern
with a low MAE of 0.34.

However, the model is overwhelmingly driven by `gsc_impressions`, and the
experiment does not establish that the model improves future refresh decisions.

The next improvement should therefore focus on obtaining a legitimate
future-window outcome/label and testing whether the model adds value beyond
the hand-written baseline.
